### Importing dependancies


In [ ]:
from typing import Tuple
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

### Defining feature groups

In [ ]:
NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges"]
BINARY_FEATURES = ["gender", "Partner", "Dependents", "PhoneService", "PaperlessBilling"]
NOMINAL_FEATURES = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod",
]
PASSTHROUGH_FEATURES = ["SeniorCitizen"]
DROP_COLUMNS = ["customerID"]
TARGET_COL = "Churn"

### Loading and splitting the data

In [ ]:
def load_and_split_data(
    csv_path: str = "telco.csv",
    test_size: float = 0.20,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Load raw dataset, enforce schema types, drop PII, and perform stratified split."""
    df = pd.read_csv(csv_path)

    # Drop non-predictive PII identifier (GDPR data minimization)
    df = df.drop(columns=DROP_COLUMNS)

    # Coerce whitespace strings in TotalCharges to NaN (imputed inside CV/train pipeline)
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    # Encode binary target
    y = df[TARGET_COL].map({"No": 0, "Yes": 1})
    X = df.drop(columns=[TARGET_COL])

    # Stratified 80/20 split BEFORE any fitting (Zero Data Leakage)
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )
    return X_train, X_test, y_train, y_test

### Adding preprocessing pipeline

### Checking the pipeline